In [1]:
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Probit
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("probit_dataset.csv")
df

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,mar_st,visit_doctor,work,alcohol,smoking,phys_active,region,is_health_good,is_health_very_good,diploma
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4593,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,142.0,0.0,0.0,0.0
4594,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,142.0,0.0,0.0,0.0
4595,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0
4596,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0


In [3]:
df.columns

Index(['heart', 'lungs', 'liver', 'kidneys', 'stomach', 'spine', 'diabetes',
       'hypertension', 'joints', 'ENT_organs', 'neurology', 'eyes', 'allergy',
       'veins', 'skin', 'oncology', 'age', 'income', 'n_child', 'sex',
       'type_area', 'invalid', 'mar_st', 'visit_doctor', 'work', 'alcohol',
       'smoking', 'phys_active', 'region', 'is_health_good',
       'is_health_very_good', 'diploma'],
      dtype='str')

In [4]:

# 1. Создаём словарь "код региона -> значение alcohol_by_region 
# (Потребление алкогольной продукции на душу населения (в литрах этанола) для этого региона)"
alcohol_dict = {
    1: 96.13,
    9: 77.48,
    10: 124.47,
    12: 121.17,
    14: 102.76,
    33: 93.26,
    39: 86.51,
    45: 100.86,
    46: 132.27,
    47: 96.62,
    48: 109.46,
    52: 49.65,
    58: 109.66,
    66: 100.53,
    67: 106.43,
    70: 100.44,
    71: 111.83,
    72: 109.08,
    73: 100.53,
    77: 28.97,
    84: 109.66,
    86: 85.72,
    89: 146.86,
    92: 99.02,
    93: 113.16,
    100: 100.44,
    105: 146.86,
    106: 120.05,
    107: 120.05,
    116: 103.08,
    117: 139.63,
    129: 77.48,
    135: 109.62,
    136: 100.89,
    137: 63.11,
    138: 60.22,
    141: 80.65,
    142: 97.76,
    161: 104.41,
    200: 117.44
}

# 2. Создаём словарь "код региона -> значение smoking_by_region 
# (Объем легальных розничных продаж сигарет на душу совершеннолетнего населения для этого региона)"
smoking_dict = {
    1: 411,
    9: 402,
    10: 307,
    12: 379,
    14: 344,
    33: 295,
    39: 278,
    45: 294,
    46: 377,
    47: 327,
    48: 275,      
    52: 248,
    58: 337,
    66: 361,
    67: 398,
    70: 311,
    71: 432,
    72: 313,
    73: 361,
    77: 80,
    84: 337,
    86: 432,      
    89: 539,      
    92: 483,
    93: 549,
    100: 311,
    105: 539,
    106: 354,
    107: 354,
    116: 384,     
    117: 300,
    129: 402,
    135: 300,
    136: 334,
    137: 301,
    138: 257,
    141: 283,
    142: 458,
    161: 323,
    200: 309
}

# 3. Создаём словарь "код региона -> значение marriages_by_region 
# (Число зарегистрированных браков в расчете на 1000 населения (оперативные данные) для этого региона)"

marriage_dict = {
    1: 3.9,
    9: 7.3,
    10: 5.4,
    12: 6.1,
    14: 5.3,
    33: 5.3,
    39: 5.5,
    45: 5.9,
    46: 5.9,
    47: 6.1,
    48: 4.4,
    52: 5.1,
    58: 6.5,
    66: 6.7,
    67: 6.0,
    70: 5.5,
    71: 6.6,
    72: 5.5,
    73: 6.7,
    77: 4.6,
    84: 6.5,
    86: 5.7,
    89: 5.9,
    92: 8.0,
    93: 7.7,
    100: 5.5,
    105: 5.9,
    106: 6.5,
    107: 6.5,
    116: 6.1,
    117: 5.4,
    129: 7.3,
    135: 5.8,
    136: 5.5,
    137: 6.0,
    138: 6.6,
    141: 9.0,
    142: 5.5,
    161: 7.1,
    200: 6.3
}


# 3. Создаём словарь "код региона -> значение phys_activity_by_region 
# (Рейтинговый балл по приверженности населения ЗОЖ для этого региона)"
phys_dict = {
    1: 60.2,
    9: 81.7,
    10: 55.2,
    12: 55.8,
    14: 72.7,
    33: 76.2,
    39: 75.7,
    45: 70.2,
    46: 59.3,
    47: 64.6,
    48: 74.8,      
    52: 81.5,
    58: 67.8,
    66: 46.7,
    67: 62.7,
    70: 68.9,
    71: 63.1,
    72: 72.5,
    73: 46.7,
    77: 81.8,
    84: 67.8,
    86: 58.9,      
    89: 54.6,      
    92: 56.9,
    93: 53.3,
    100: 68.9,
    105: 54.6,
    106: 43.2,
    107: 43.2,
    116: 66.2,     
    117: 76.0,
    129: 81.7,
    135: 69.6,
    136: 81.5,
    137: 74.9,
    138: 76.0,
    141: 79.3,
    142: 80.1,
    161: 60.9,
    200: 60.1
}


df['alcohol_by_region'] = df['region'].map(alcohol_dict)
df['smoking_by_region'] = df['region'].map(smoking_dict)
df['marriages_by_region'] = df['region'].map(marriage_dict)
df['phys_activity_by_region'] = df['region'].map(phys_dict)

df

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,smoking,phys_active,region,is_health_good,is_health_very_good,diploma,alcohol_by_region,smoking_by_region,marriages_by_region,phys_activity_by_region
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,96.13,411,3.9,60.2
1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,96.13,411,3.9,60.2
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,96.13,411,3.9,60.2
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,96.13,411,3.9,60.2
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,96.13,411,3.9,60.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4593,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,142.0,0.0,0.0,0.0,97.76,458,5.5,80.1
4594,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,142.0,0.0,0.0,0.0,97.76,458,5.5,80.1
4595,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,0.0,77.0,0.0,0.0,0.0,28.97,80,4.6,81.8
4596,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,77.0,0.0,0.0,0.0,28.97,80,4.6,81.8


In [5]:
df.columns

Index(['heart', 'lungs', 'liver', 'kidneys', 'stomach', 'spine', 'diabetes',
       'hypertension', 'joints', 'ENT_organs', 'neurology', 'eyes', 'allergy',
       'veins', 'skin', 'oncology', 'age', 'income', 'n_child', 'sex',
       'type_area', 'invalid', 'mar_st', 'visit_doctor', 'work', 'alcohol',
       'smoking', 'phys_active', 'region', 'is_health_good',
       'is_health_very_good', 'diploma', 'alcohol_by_region',
       'smoking_by_region', 'marriages_by_region', 'phys_activity_by_region'],
      dtype='str')

In [6]:

sexes = [1, 2]
illnesses = ['heart', 'lungs', 'liver', 'kidneys', 'stomach', 'spine', 'diabetes',
       'hypertension', 'joints', 'ENT_organs', 'neurology', 'eyes', 'allergy',
       'veins', 'skin', 'oncology']

for sex in sexes:
    for illness in illnesses:

        df_sex = df[df['sex'] == sex].copy()  

        # Исход
        outcome = illness

        # Эндогенные переменные:
        endog_vars = ['diploma', 'mar_st', 'alcohol', 'smoking', 'phys_active']

        # Экзогенные переменные:
        exog_vars = ['age', 'income', 'n_child', 'type_area', 'invalid', 'visit_doctor', 'work']

        # Инструменты для каждого эндогенного регрессора
        # (переменные, которые НЕ входят в уравнение здоровья)
        instruments = {
            'diploma': [],  
            'mar_st': ['marriages_by_region'],
            'alcohol': ['alcohol_by_region'],
            'smoking': ['smoking_by_region'],
            'phys_active': ['phys_activity_by_region'] 
        }

       
        for var in endog_vars:
            
            instr_for_var = instruments.get(var, [])
            
            first_step_vars = exog_vars + instr_for_var
            
            X_first = sm.add_constant(df_sex[first_step_vars])
            y_first = df_sex[var]
            
            model_first = Probit(y_first, X_first)
            result_first = model_first.fit(disp=0)
            

            df_sex[f'{var}_hat'] = result_first.predict(X_first)

        
        X_second_vars = exog_vars + [f'{v}_hat' for v in endog_vars]
        X_second = sm.add_constant(df_sex[X_second_vars])
        y_second = df_sex[outcome]

        # Оцениваем пробит с предсказанными значениями
        model_second = Probit(y_second, X_second)
        result_second = model_second.fit()

        # Результаты
        print()
        print(f"Результаты для {sex} и {illness}")
        print()
        print(result_second.summary())

        # Коэффициент для diploma (эффект образования на здоровье сердца)
        print(f"\nЭффект высшего образования: {result_second.params['diploma_hat']:.4f}, p-value: {result_second.pvalues['diploma_hat']:.4f}")

        print("_._."*50)
        print()
        print("_._."*50)

Optimization terminated successfully.
         Current function value: 0.182293
         Iterations 7

Результаты для 1 и heart

                          Probit Regression Results                           
Dep. Variable:                  heart   No. Observations:                 1926
Model:                         Probit   Df Residuals:                     1914
Method:                           MLE   Df Model:                           11
Date:                Sun, 03 May 2026   Pseudo R-squ.:                  0.1263
Time:                        22:48:15   Log-Likelihood:                -351.10
converged:                       True   LL-Null:                       -401.83
Covariance Type:            nonrobust   LLR p-value:                 9.161e-17
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
age                 0.0070      0.027      0.261      0.794      -0.045

c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


                          Probit Regression Results                           
Dep. Variable:                  heart   No. Observations:                 2672
Model:                         Probit   Df Residuals:                     2660
Method:                           MLE   Df Model:                           11
Date:                Sun, 03 May 2026   Pseudo R-squ.:                  0.1126
Time:                        22:48:17   Log-Likelihood:                -409.77
converged:                       True   LL-Null:                       -461.76
Covariance Type:            nonrobust   LLR p-value:                 2.898e-17
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
age                 0.0647      0.030      2.148      0.032       0.006       0.124
income          -1.949e-05   2.13e-05     -0.917      0.359   -6.11e-05    2.22e-05
n_child             0.0866      

In [7]:
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import Probit
import pandas as pd
import numpy as np

sexes = [1, 2]
illnesses = ['heart', 'lungs', 'liver', 'kidneys', 'stomach', 'spine', 'diabetes',
       'hypertension', 'joints', 'ENT_organs', 'neurology', 'eyes', 'allergy',
       'veins', 'skin', 'oncology']

# Словарь для хранения результатов
results_dict = []

for sex in sexes:
    for illness in illnesses:

        df_sex = df[df['sex'] == sex].copy()  
        
        # Исход
        outcome = illness
        
        # Эндогенные переменные (те, что могут зависеть от ненаблюдаемых факторов)
        endog_vars = ['diploma', 'mar_st', 'alcohol', 'smoking', 'phys_active']
        
        # Экзогенные переменные (контрольные, которые включаем везде)
        exog_vars = ['age', 'income', 'n_child', 'type_area', 'invalid', 'visit_doctor', 'work']
        
        # Инструменты для каждого эндогенного регрессора
        instruments = {
            'diploma': ['alcohol_by_region', 'smoking_by_region', 'marriages_by_region', 'phys_activity_by_region'],
            'mar_st': ['marriages_by_region'],
            'alcohol': ['alcohol_by_region'],
            'smoking': ['smoking_by_region'],
            'phys_active': ['phys_activity_by_region'] 
        }
        
        # ---- ШАГ 1: Редуцированные формы для каждой эндогенной переменной ----
        resid_cols = []
        
        for var in endog_vars:
            instr_for_var = instruments.get(var, [])
            first_step_vars = exog_vars + instr_for_var
            
            X_first = sm.add_constant(df_sex[first_step_vars])
            y_first = df_sex[var]
            
            model_first = Probit(y_first, X_first)
            result_first = model_first.fit(disp=0)
            
            # Предсказанная вероятность
            pred_prob = result_first.predict(X_first)
            
            # ***** ГЛАВНОЕ ИЗМЕНЕНИЕ: считаем residuals (обобщенные остатки) *****
            # Для пробит: обобщенный остаток = y - Pr(y=1) 
            # (более правильно: использовать обобщенные остатки, но для учебных целей сойдет)
            resid = y_first - pred_prob
            resid_name = f'{var}_resid'
            df_sex[resid_name] = resid
            resid_cols.append(resid_name)
            
            # (Опционально) Сохраняем предсказанные значения, если нужно
            df_sex[f'{var}_pred'] = pred_prob
        
        # ---- ШАГ 2: Структурная модель здоровья с residuals ----
        # Включаем:
        # 1) Исходные эндогенные переменные (прямой эффект)
        # 2) Экзогенные переменные
        # 3) *** RESIDUALS *** (это ключевое отличие!)
        
        X_second_vars = exog_vars + endog_vars + resid_cols
        X_second = sm.add_constant(df_sex[X_second_vars])
        y_second = df_sex[outcome]
        
        # Удаляем строки с пропусками (если есть)
        valid_idx = ~(y_second.isna() | X_second.isna().any(axis=1))
        X_second_clean = X_second[valid_idx]
        y_second_clean = y_second[valid_idx]
        
        # Оцениваем пробит
        model_second = Probit(y_second_clean, X_second_clean)
        result_second = model_second.fit(disp=0)
        
        # ---- Интерпретация результатов ----
        # Коэффициент на diploma - это и есть эффект образования на здоровье
        # (правильный, очищенный от эндогенности)
        
        coef_diploma = result_second.params.get('diploma', np.nan)
        pval_diploma = result_second.pvalues.get('diploma', np.nan)
        
        # Тест на экзогенность: если residuals значимы -> эндогенность есть
        resid_diploma_pval = result_second.pvalues.get('diploma_resid', np.nan)
        
        # Сохраняем результаты
        results_dict.append({
            'sex': sex,
            'illness': illness,
            'coef_diploma': coef_diploma,
            'pval_diploma': pval_diploma,
            'resid_diploma_pval': resid_diploma_pval,
            'n_obs': len(y_second_clean),
            'pseudo_r2': result_second.prsquared
        })
        
        # Печать результатов
        print(f"\n{'='*60}")
        print(f"Пол: {'Мужчины' if sex==1 else 'Женщины'}, Заболевание: {illness}")
        print(f"{'='*60}")
        print(f"Эффект высшего образования: {coef_diploma:.4f}, p-value: {pval_diploma:.4f}")
        print(f"Тест на эндогенность (residual значим?): p = {resid_diploma_pval:.4f}")
        if resid_diploma_pval < 0.05:
            print("→ Эндогенность присутствует, контрольная функция нужна!")
        else:
            print("→ Эндогенность статистически не значима")
        print(f"Количество наблюдений: {len(y_second_clean)}")
        print()

# ---- Сводная таблица результатов ----
results_df = pd.DataFrame(results_dict)
print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*80)
print(results_df.pivot_table(index='illness', columns='sex', values='coef_diploma', 
                              aggfunc='first', fill_value='-'))


Пол: Мужчины, Заболевание: heart
Эффект высшего образования: -1.2293, p-value: 0.4110
Тест на эндогенность (residual значим?): p = 0.3296
→ Эндогенность статистически не значима
Количество наблюдений: 1926


Пол: Мужчины, Заболевание: lungs
Эффект высшего образования: -3.6877, p-value: 0.0258
Тест на эндогенность (residual значим?): p = 0.0307
→ Эндогенность присутствует, контрольная функция нужна!
Количество наблюдений: 1926


Пол: Мужчины, Заболевание: liver
Эффект высшего образования: -1.0075, p-value: 0.4936
Тест на эндогенность (residual значим?): p = 0.4333
→ Эндогенность статистически не значима
Количество наблюдений: 1926


Пол: Мужчины, Заболевание: kidneys
Эффект высшего образования: -0.0854, p-value: 0.9536
Тест на эндогенность (residual значим?): p = 0.9903
→ Эндогенность статистически не значима
Количество наблюдений: 1926


Пол: Мужчины, Заболевание: stomach
Эффект высшего образования: -0.1154, p-value: 0.9180
Тест на эндогенность (residual значим?): p = 0.9690
→ Эндоген

c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Huaweii\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "



Пол: Мужчины, Заболевание: oncology
Эффект высшего образования: 5.9854, p-value: 0.5153
Тест на эндогенность (residual значим?): p = 0.5393
→ Эндогенность статистически не значима
Количество наблюдений: 1926


Пол: Женщины, Заболевание: heart
Эффект высшего образования: -0.9962, p-value: 0.5058
Тест на эндогенность (residual значим?): p = 0.5205
→ Эндогенность статистически не значима
Количество наблюдений: 2672


Пол: Женщины, Заболевание: lungs
Эффект высшего образования: -0.8348, p-value: 0.5519
Тест на эндогенность (residual значим?): p = 0.7370
→ Эндогенность статистически не значима
Количество наблюдений: 2672


Пол: Женщины, Заболевание: liver
Эффект высшего образования: 0.4058, p-value: 0.8234
Тест на эндогенность (residual значим?): p = 0.7934
→ Эндогенность статистически не значима
Количество наблюдений: 2672


Пол: Женщины, Заболевание: kidneys
Эффект высшего образования: -0.2436, p-value: 0.8264
Тест на эндогенность (residual значим?): p = 0.9889
→ Эндогенность статистичес